# Занятие 2. LangGraph: состояние, узлы, ребра

**Цикл агента вы описываете графом, а выполняет его LangGraph.**

- агент делает то же, что на занятии 1; LangGraph добавляет паузы, чекпоинты, трейсы и подграфы
- модель создается в одном месте через `init_chat_model`: в ноутбуках нет ни имени модели, ни ключа
- температура нигде не задается: текущие модели Gemini могут зацикливаться, если ее менять


In [ ]:
# Собрать объект модели из .env через init_chat_model и функцию, которая рисует граф картинкой.

import operator
import os
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from IPython.display import Image, display
from langchain.chat_models import init_chat_model

load_dotenv()  # читает .env один раз; ниже файлы никто не открывает


def chat_model(size: str = "cheap", **kwargs):
    """A model object for the configured provider. A dozen lines, copy them once."""
    name = os.environ[f"MODEL_{size.upper()}"]  # имена моделей живут в .env, а не в коде
    secret = os.environ["LLM_API_KEY"]
    if os.getenv("LLM_REASONING_EFFORT"):  # gpt-5.x: инструментам нужен reasoning "none"
        kwargs.setdefault("reasoning_effort", os.environ["LLM_REASONING_EFFORT"])
    kwargs.setdefault("timeout", 120)  # каждая попытка обрывается через 2 минуты; без этого висит вечно
    kwargs.setdefault("max_tokens", 8192)  # ограничивает ответ, который зациклился, например на пробелах
    # Gemini через OpenAI-compat теряет подпись рассуждений: на втором витке 400
    if os.getenv("LLM_PROVIDER", "openai_compat") == "google_genai":
        return init_chat_model(f"google_genai:{name}", api_key=secret, **kwargs)
    return init_chat_model(
        f"openai:{name}", api_key=secret, base_url=os.environ["LLM_BASE_URL"], **kwargs
    )


def draw(graph) -> None:
    """Show a compiled graph as a picture, or as Mermaid text when offline."""
    try:  # картинку рисует mermaid.ink; туда уходят только имена узлов
        display(Image(graph.get_graph().draw_mermaid_png()))
    except Exception:  # нет сети: тот же граф текстом
        print(graph.get_graph().draw_mermaid())


print("provider:", os.getenv("LLM_PROVIDER", "openai_compat"),
      "| strong:", os.environ["MODEL_STRONG"])


## Чего не умеет ручной цикл

**Цикл из занятия 1 рассчитан на один инструмент.**

- второй инструмент: разбирать имя инструмента, ветвиться по нему, вести таблицу «имя → функция»
- паузы с продолжением нет: состояние прогона хранится в локальных переменных цикла `for` и теряется при выходе из функции
- шаги прогона можно увидеть только через `print`: он не показывает, какой шаг работал, что модель получила на третьем витке и сколько работал инструмент
- на выдуманное имя `careNotes` парсер возвращает `None`, как и на готовый ответ


In [ ]:
# Расширить парсер занятия 1 до двух инструментов: меняется тип результата, проскакивают выдуманные имена.

QUOTES = "'" + '"'  # кавычки вокруг аргумента модель ставит не на каждом витке


# один инструмент: имя понятно и так, его не возвращают
def parse_action_one(text: str) -> str | None:
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("Action:") and "(" in line:
            return line[line.index("(") + 1 : line.rindex(")")].strip().strip(QUOTES)
    return None  # строки Action нет: модель ответила вместо вызова


# два инструмента: меняется тип результата, а за ним и весь вызывающий код
def parse_action_two(text: str) -> tuple[str, str] | None:
    for line in text.splitlines():
        line = line.strip()
        if not line.startswith("Action:") or "(" not in line:
            continue
        body = line[len("Action:") :].strip()
        name = body[: body.index("(")].strip()
        argument = body[body.index("(") + 1 : body.rindex(")")].strip().strip(QUOTES)
        if name not in ("find_species", "care_notes"):
            return None  # модель выдумала инструмент. И что теперь?
        return name, argument
    return None


print(parse_action_two('Action: care_notes("mon-01")'))
print(parse_action_two('Action: careNotes("mon-01")'))


## Сборка графа: состояние

**Все узлы работают с одним словарем состояния, и каждый узел возвращает только измененные ключи.**

- редьюсер — функция, которую задают полю в схеме: она получает старое значение и новое от узла и возвращает то, что попадет в состояние
- у поля без редьюсера новое значение заменяет старое
- схему получает граф, а не узлы
- сегодня каждый `invoke` получает `recursion_limit`; зачем, объясняется дальше в ноутбуке


In [ ]:
# Собрать граф из одного узла без редьюсера у поля messages: одно обновление заменит всю историю.

from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import END, START, StateGraph


class Broken(TypedDict):
    messages: list  # редьюсера нет, поэтому по умолчанию замена


def add_reply(state) -> dict:
    return {"messages": [AIMessage("the reply")]}  # в историю даже не заглядывает


builder = StateGraph(Broken)
builder.add_node("add_reply", add_reply)
builder.add_edge(START, "add_reply")  # START и END — два встроенных узла
builder.add_edge("add_reply", END)

result = builder.compile().invoke(
    {"messages": [HumanMessage("the question")]},
    config={"recursion_limit": 5},
)
print([m.content for m in result["messages"]])


**Без редьюсера ответ заменяет историю, и вопрос пропадает без ошибки и предупреждения.**

- в состоянии осталось одно сообщение, ответ
- самая частая ошибка занятия: похоже на забывчивую модель, но сначала проверьте редьюсер
- `add_messages` — готовый редьюсер для сообщений: дописывает новые и сверяет их по id, поэтому повторно переданная история не задваивается


In [ ]:
# Пересобрать тот же граф с add_messages у поля: теперь обновление узла дописывается.

from langgraph.graph.message import add_messages


class Fixed(TypedDict):
    messages: Annotated[list, add_messages]  # reducer(old, update) -> new; дописывает


builder = StateGraph(Fixed)
builder.add_node("add_reply", add_reply)  # та же функция узла, без изменений
builder.add_edge(START, "add_reply")
builder.add_edge("add_reply", END)

result = builder.compile().invoke(
    {"messages": [HumanMessage("the question")]},
    config={"recursion_limit": 5},
)
print([m.content for m in result["messages"]])


In [ ]:
# Напечатать готовую аннотацию из MessagesState: тот же редьюсер, уже объявленный за вас.

from langgraph.graph import MessagesState

print(MessagesState.__annotations__["messages"])  # это поле больше не пишут руками


### Узлы и ребра

**Узел — функция, которая получает состояние и возвращает обновление, а ребро задает, какой узел запускается следующим.**

- в следующей ячейке модели нет: узел `shout` переводит текст в верхний регистр, потом узел `frame` берет его в `<< >>`
- у `text` редьюсера нет, и каждый узел его заменяет; у `trail` редьюсер `operator.add`, и каждый узел дописывает туда свое имя
- в конце в `text` результат последнего узла, а в `trail` список узлов в том порядке, в каком они работали
- редьюсер может быть у любого поля, не только у `messages`


In [ ]:
# Связать две обычные функции в граф с двумя полями: одно заменяется, другое копит редьюсер.

class Note(TypedDict):
    text: str  # редьюсера нет: каждая запись заменяет значение
    trail: Annotated[list[str], operator.add]  # есть редьюсер, поэтому записи склеиваются


def shout(state: Note) -> dict:
    return {"text": state["text"].upper(), "trail": ["shout"]}


def frame(state: Note) -> dict:
    return {"text": f"<<{state['text']}>>", "trail": ["frame"]}


builder = StateGraph(Note)
builder.add_node("shout", shout)  # обычная функция под именем
builder.add_node("frame", frame)
builder.add_edge(START, "shout")
builder.add_edge("shout", "frame")
builder.add_edge("frame", END)
notes = builder.compile()  # до compile ничего не запускается; compile проверяет связи

draw(notes)  # функция из ячейки 1

print(
    notes.invoke(
        {"text": "a graph is a dict and some functions", "trail": []},  # начальное состояние: текст и пустой trail
        config={"recursion_limit": 5},
    )
)


**Ключ, которого нет в схеме, пропадает без ошибки.**

- в следующей ячейке `frame` пишет `trial` вместо `trail`: узел отработал, а в `trail` одна запись
- `TypedDict` — подсказка для проверки типов, а не проверка во время выполнения
- если поле так и не заполнилось, найдите ключ через grep и посчитайте варианты написания


In [ ]:
# Запустить граф, где узел пишет ключ состояния с опечаткой: запись пропадет без ошибки.

def frame_typo(state: Note) -> dict:
    return {"text": f"<<{state['text']}>>", "trial": ["frame"]}  # trial вместо trail


builder = StateGraph(Note)
builder.add_node("shout", shout)
builder.add_node("frame", frame_typo)  # единственное отличие от прошлого графа
builder.add_edge(START, "shout")
builder.add_edge("shout", "frame")
builder.add_edge("frame", END)

print(
    builder.compile().invoke(
        {"text": "one letter", "trail": []},
        config={"recursion_limit": 5},
    )
)


### Модель в узле

**`care_notes` принимает id, полученный от `find_species`, поэтому нужен второй виток.**

- `bind_tools` прикрепляет JSON-схемы инструментов, как `tools=` в конце занятия 1; откуда они берутся, покажет занятие 3
- модель читает докстроки: у `care_notes` сказано, что id берется из `find_species`, и это задает порядок вызовов


In [ ]:
# Объявить через @tool два зависимых инструмента про растения: второму нужен id, который вернет первый.

from langchain_core.tools import tool

# поддельная база: без сети, каждый прогон дает одно и то же
SPECIES = {"monstera": "mon-01", "ficus": "fic-02", "basil": "bas-03", "cactus": "cac-04"}
CARE = {
    "mon-01": "Water every 7 days. Bright indirect light. Wipe the leaves monthly.",
    "fic-02": "Water every 10 days. Hates being moved. Drops leaves when it sulks.",
    "bas-03": "Water daily. Full sun. Pinch the flowers off to keep the leaves coming.",
    "cac-04": "Water every 21 days in summer, never in winter. Full sun.",
}


@tool  # докстрока ниже — это описание, которое читает модель
def find_species(name: str) -> str:
    """Look up the species id of a house plant by its common name."""
    key = name.strip().lower()
    if key not in SPECIES:
        # ошибка, с которой модель может работать, а не исключение
        return f"Unknown plant {name!r}. Known names: {', '.join(sorted(SPECIES))}."
    return SPECIES[key]


@tool
def care_notes(species_id: str) -> str:
    """Return watering and light instructions for a species id from find_species."""
    if species_id not in CARE:
        # на выдуманный id вернется эта строка; модель может повторить на следующем витке
        return f"No care notes for {species_id!r}. Call find_species first to get a valid id."
    return CARE[species_id]


TOOLS = [find_species, care_notes]
print([t.name for t in TOOLS])  # имя инструмента — это имя функции


In [ ]:
# Привязать оба инструмента и задать один вопрос: вернется вызов, который еще никто не выполнил.

QUESTION = "My monstera looks sad. How often should I be watering it?"

model = chat_model("strong")
bound = model.bind_tools(TOOLS)  # новый объект; здесь ничего не отправляется

ai = bound.invoke([{"role": "user", "content": QUESTION}])
print("type:      ", type(ai).__name__)
print("content:   ", repr(ai.content))  # может быть пусто или короткая реплика при вызове
print("tool_calls:", ai.tool_calls)  # id свяжет результат с этим вызовом


**Ответ модели — запрос на вызов инструмента с id; инструмент еще не запущен.**

- один запрос, один виток; обычный словарь принимается вместо `HumanMessage`
- результат инструмента должен вернуть тот же id в поле `tool_call_id`


### Условное ребро руками

**Условное ребро — обычная функция: на входе состояние, на выходе имя следующего узла.**

- `should_continue` занимает пять строк: `"tools"`, если в последнем сообщении есть `tool_calls`, иначе `END`
- список направлений рисует ребра и ограничивает маршрут: имя вне списка дает `KeyError`
- без небольшого `recursion_limit` зациклившийся граф продолжает слать платные запросы


In [ ]:
# Собрать цикл инструментов руками: узел модели, ToolNode и условное ребро в пять строк между ними.

from langgraph.prebuilt import ToolNode


def call_model(state: MessagesState) -> dict:
    # Верните только новый ответ: add_messages допишет его в историю.
    return {"messages": [bound.invoke(state["messages"])]}


def should_continue(state: MessagesState) -> str:
    last = state["messages"][-1]
    if last.tool_calls:  # все решение о маршруте — один атрибут
        return "tools"
    return END  # тоже допустимое направление, и оно завершает прогон


builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_node("tools", ToolNode(TOOLS))  # один узел, все инструменты, выбор по имени
builder.add_edge(START, "model")
builder.add_conditional_edges("model", should_continue, ["tools", END])  # честный рисунок
builder.add_edge("tools", "model")  # это ребро, и только оно, замыкает цикл
manual = builder.compile()

draw(manual)

result = manual.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    config={"recursion_limit": 8},  # в langgraph 1.2 по умолчанию 10007, а не 25
)
print(len(result["messages"]), "messages")
print(result["messages"][-1].content)


**`tools_condition` делает то же, что ваш маршрутизатор, но требует узел с именем `tools`.**

- вместо своей функции вы пишете один импорт
- в документации LangGraph он не упоминается ни на одной странице; в быстром старте маршрутизатор написан руками
- в примере из его докстроки два импорта, и оба падают с `ImportError`


In [ ]:
# Заменить ручной маршрутизатор готовым tools_condition и скомпилировать граф для остальных ячеек.

from langgraph.prebuilt import tools_condition

builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
# tools_condition возвращает Literal["tools", "__end__"], поэтому имя узла только такое
builder.add_node("tools", ToolNode(TOOLS))
builder.add_edge(START, "model")
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("tools", "model")
graph = builder.compile()  # этот граф работает до конца ноутбука

draw(graph)


**Если переименовать узел `tools`, `tools_condition` сломается.**

- он по-прежнему возвращает `"tools"`, а узла с таким именем нет
- `compile()` бросает `ValueError` еще до первого платного запроса


In [ ]:
# Переименовать узел инструментов, оставив tools_condition: компиляция упадет, узла tools нет.

builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_node("plant_tools", ToolNode(TOOLS))  # единственное изменение в ячейке
builder.add_edge(START, "model")
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("plant_tools", "model")  # ребро цикла в порядке, маршрутизатор — нет

try:
    builder.compile()
except ValueError as error:
    print("ValueError:", error)


### ToolNode и весь цикл

**`ToolNode` одной строкой заменяет ваш код вызова инструментов из занятия 1.**

- для каждого вызова из `tool_calls` последнего ответа запускает инструмент по имени и возвращает `ToolMessage` с id вызова


In [ ]:
# Прогнать весь цикл один раз и напечатать все сообщения: проследить id каждого вызова до результата.

result = graph.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    config={"recursion_limit": 8},
)

for message in result["messages"]:  # роль, вызов и id, на который отвечает сообщение
    message.pretty_print()


**Два витка цикла дали шесть сообщений; оба витка выполнил LangGraph.**

- 1 человек; 2 модель: `find_species("monstera")`; 3 инструмент: `mon-01`
- 4 модель: `care_notes("mon-01")`, аргумент взят из сообщения 3
- 5 инструмент: инструкции по уходу; 6 модель: ответ без вызовов, дальше `END`
- один id вызова связывает 2 и 3, другой — 4 и 5; эти пары видны в каждом трейсе


**Вызов инструмента внутри `ToolNode` может не сработать по двум причинам, и обрабатываются они по-разному.**

- на выдуманное имя инструмента `ToolNode` возвращает `ToolMessage` со списком допустимых имен, исключения нет
- в полном цикле модель может исправить имя на следующем витке; тестовый граф ниже идет в `END`


In [ ]:
# Передать ToolNode вызов несуществующего инструмента: ошибка вернется сообщением.

def hallucinate(state: MessagesState) -> dict:
    # поддельный узел модели: запрос никуда не уходит
    return {
        "messages": [
            AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": "findSpecies",  # такого инструмента нет
                        "args": {"name": "monstera"},
                        "id": "call-1",  # обычно его ставит провайдер
                    }
                ],
            )
        ]
    }


builder = StateGraph(MessagesState)
builder.add_node("model", hallucinate)
builder.add_node("tools", ToolNode(TOOLS))
builder.add_edge(START, "model")
builder.add_edge("model", "tools")  # обычное ребро: прямо в инструменты
builder.add_edge("tools", END)

out = builder.compile().invoke({"messages": []}, config={"recursion_limit": 5})
print(out["messages"][-1].content)


In [ ]:
# Зарегистрировать инструмент, который бросает исключение: ToolNode пробросит его и оборвет весь прогон.

from langchain_core.tools import tool


@tool
def flaky(species_id: str) -> str:
    """Looks like any other tool right up until the day it does not."""
    raise TimeoutError("upstream took too long")  # заглушка вместо HTTP-вызова


def call_flaky(state: MessagesState) -> dict:
    # та же поддельная модель, нацеленная на падающий инструмент
    return {
        "messages": [
            AIMessage(
                content="",
                tool_calls=[
                    {"name": "flaky", "args": {"species_id": "mon-01"}, "id": "call-1"}
                ],
            )
        ]
    }


builder = StateGraph(MessagesState)
builder.add_node("model", call_flaky)
builder.add_node("tools", ToolNode([flaky]))  # зарегистрирован лишь падающий инструмент
builder.add_edge(START, "model")
builder.add_edge("model", "tools")
builder.add_edge("tools", END)

try:
    builder.compile().invoke({"messages": []}, config={"recursion_limit": 5})
except TimeoutError as error:
    print("the run is over:", type(error).__name__, error)  # ToolNode пробросил его


**Исключение в инструменте обрывает прогон; ловите его в инструменте и возвращайте текстом.**

- `ToolNode` пробрасывает все исключения, кроме собственной ошибки вызова
- один таймаут обрывает весь прогон, и собранные сообщения теряются
- возвращайте строку с подсказкой, что попробовать, как `find_species` для неизвестного растения


## init_chat_model

**`init_chat_model` создает объект модели из строки «провайдер:модель», собранной по `.env`.**

- тот же `.env`, те же `MODEL_CHEAP` и `MODEL_STRONG`, что на занятии 1
- возвращается объект модели с `bind_tools`, а не клиент, которому отправляют словари
- от размера зависит только то, какое имя модели читается из `.env`
- `ai` — объект ответа, полученный выше в этом ноутбуке


In [ ]:
# Создать обе модели через фабрику и вывести поля, которые провайдер добавляет к своему ответу.

cheap = chat_model("cheap")
strong = chat_model("strong")
print(type(cheap).__name__, "|", type(strong).__name__)  # оба раза один класс

print("id:               ", ai.id)  # от LangChain; id провайдера — в response_metadata
print("additional_kwargs:", list(ai.additional_kwargs))  # особые поля провайдера
print("response_metadata:", list(ai.response_metadata))  # причина остановки, токены


**Дописывайте ответ модели как есть: пересобранная копия теряет id и метаданные.**

- провайдеры добавляют к объекту ответа подписи рассуждений, маркеры кэша и id
- копия, собранная из `content` и `tool_calls`, печатается так же, но у части провайдеров следующий запрос падает
- копия теряет `id`, `additional_kwargs` и `response_metadata`


## Практика

**Перенесите ассистента на граф в своем репозитории.**

1. `MessagesState`, узел модели, `ToolNode`, одно условное ребро
2. `should_continue` руками, затем замена на `tools_condition`
3. инструмент из занятия 1 становится `@tool` с настоящей докстрокой: модель читает именно этот текст
4. удалите `parse_action` и промпт с форматом, а не закомментируйте


**Второй инструмент должен брать результат первого, иначе `tools` не отработает второй раз.**

5. второй и третий инструмент, один берет результат другого: поиск, который возвращает id, и инструмент, который этот id принимает
6. каждый `invoke` передает небольшой `recursion_limit`
7. закоммитьте `runs/session-02.md`; что туда входит, перечислено в карточке обязательного артефакта после раздела про Langfuse
8. спрашивайте сразу: три сегодняшних сбоя похожи на ошибки модели

Ноутбук для справки: домен у вас другой, и его код один в один не перенесется.

## Наблюдаемость

**Трейс показывает, какой узел работал, что получила модель и сколько длился шаг.**

- Langfuse работает в Docker на вашем компьютере: без регистрации, выбора региона и квот
- трейс на прогон, спан на узел, список сообщений, отправленный модели, длительность каждого шага
- трейсы не покидают ваш компьютер


Рядом с вашим `.env`:

```
curl -sSLO https://raw.githubusercontent.com/langfuse/langfuse/v3.224.0/docker-compose.yml
```

Затем положите рядом `docker-compose.override.yml`; compose подхватит его без `-f`.

```yaml
services:
  langfuse-worker:
    image: docker.io/langfuse/langfuse-worker:3.224.0
  langfuse-web:
    image: docker.io/langfuse/langfuse:3.224.0
    environment:
      LANGFUSE_INIT_ORG_ID: agents-course
      LANGFUSE_INIT_ORG_NAME: LLM Agents Course
      LANGFUSE_INIT_PROJECT_ID: assistant
      LANGFUSE_INIT_PROJECT_NAME: My assistant
      LANGFUSE_INIT_PROJECT_PUBLIC_KEY: ${LANGFUSE_PUBLIC_KEY}
      LANGFUSE_INIT_PROJECT_SECRET_KEY: ${LANGFUSE_SECRET_KEY}
      LANGFUSE_INIT_USER_EMAIL: ${LANGFUSE_USER_EMAIL}
      LANGFUSE_INIT_USER_NAME: Student
      LANGFUSE_INIT_USER_PASSWORD: ${LANGFUSE_USER_PASSWORD}
  postgres:                       # собственная база langfuse, не та, что вы добавите позже
    environment:
      POSTGRES_PASSWORD: ${LANGFUSE_POSTGRES_PASSWORD:-postgres}
```

- в исходном compose стоит тег `:3`, который переходит на новые выпуски; override закрепляет оба сервиса на проверенной версии 3.224.0
- последние две строки важны с занятия 8: без них `POSTGRES_PASSWORD`,
  который вы добавите для своей базы, попадет и в базу langfuse, а ее том уже создан
- пять новых строк в `.env`: `LANGFUSE_HOST=http://localhost:3000`, ваш email и еще три строки, которые вы придумаете сами, для двух ключей и пароля
- префиксы `pk-lf-` и `sk-lf-` сохраните
- не берите значения `LANGFUSE_INIT_*` в кавычки: кавычки станут частью значения, инициализация ничего не сделает и не покажет ошибки, а пустой Langfuse отвергнет ваши ключи

```
docker compose up -d
```

Первый запуск скачивает образы и занимает время; следующие запуски быстрые.

- compose и ваш код читают один и тот же `.env`, поэтому ключи в контейнере и в коде совпадают

**Один обработчик в config записывает трейс всего прогона, а граф не меняется.**

- печатайте хост, но никогда не ключ: код попадает в коммит
- нет трейса второго прогона? ключи в `.env` разошлись с ключами контейнера; `check_env.py` их проверит


In [ ]:
# Запустить тот же граф с обработчиком Langfuse в callbacks: трейс появится без правки узлов.

import os

from langfuse import get_client
from langfuse.langchain import CallbackHandler

client = get_client()  # берет LANGFUSE_HOST и оба ключа из окружения
print("server:", os.getenv("LANGFUSE_HOST"), "| up:", client.auth_check())

handler = CallbackHandler()  # одного обработчика хватит; каждый invoke — свой трейс
traced = graph.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    config={"recursion_limit": 8, "callbacks": [handler]},
)

client.flush()  # отправить сразу, а не с очередным пакетом через 5 секунд

print(traced["messages"][-1].content)

**Найдите в трейсе три вещи и назовите каждую вслух.**

- два спана `tools`, в каждом по одному результату инструмента
- второй вызов модели получает 3 сообщения: история дописана, а не заменена
- счетчики токенов и время каждого шага; к ним вернется занятие 3


**Обязательный артефакт: `runs/session-02.md` в коммите.**

- схема вашего графа текстом, из `graph.get_graph().draw_mermaid()`: GitHub рисует ее в блоке `mermaid`
- диалог из шести и более сообщений, где сообщение 4 использует значение из сообщения 3
- выгруженный трейс, где виден проход по узлам и вызов инструмента

## Обсуждение

**Граф длиннее цикла `for`. Когда нужен граф, а когда хватит цикла?**

- сначала защитите цикл: один вызов модели, без ветвлений
- потом разберите случаи: два инструмента и больше, пауза и продолжение, код читает кто-то еще
- в каждом случае: кто выигрывает и почему?


---

## Сегодня не разбирается

**Условное ребро может выбирать путь по любому полю состояния.**

- например, по полю, которое записал узел-классификатор
- это `should_continue` с большим числом возвращаемых значений
- для чтения дома; на занятии 9 его сравнивают с четырьмя другими схемами оркестрации


In [ ]:
# Набросать маршрутизатор по обычному полю состояния: читать дома, поэтому подключение закомментировано.

class Routed(TypedDict):
    text: str
    kind: str  # пишет classify, читает route
    reply: str


def classify(state: Routed) -> dict:
    text = state["text"].lower()
    if text.endswith("?"):
        return {"kind": "question"}
    if text.startswith(("do ", "make ", "send ")):
        return {"kind": "command"}
    return {"kind": "other"}


def route(state: Routed) -> str:
    return state["kind"]  # та же сигнатура, что у should_continue


# builder.add_conditional_edges("classify", route, ["question", "command", "other"])


## Сегодня в одной карточке

**Цикл вы описываете графом; LangGraph выполняет его и добавляет трейсы, чекпоинты и паузы.**

**Теперь вы можете объяснить:**
- узел возвращает только измененные ключи; дописать или заменить, решает редьюсер поля, а `messages` без `add_messages` затирается
- `ToolNode` делает то же, что ваш код на занятии 1: читает `tool_calls`, запускает каждый инструмент, возвращает `ToolMessage` с id вызова
- дописывайте ответ модели как есть; пересобранная копия теряет id и метаданные провайдера, и у части провайдеров следующий запрос падает

**В репозитории:** `runs/session-02.md`, mermaid-диаграмма вашего графа, диалог, где четвертое сообщение опирается на третье, один выгруженный трейс.
**Ловушка дня:** без редьюсера история заменяется, а ключ состояния с опечаткой пропадает; ошибки нет ни в том, ни в другом случае, и кажется, что виновата модель.
**Спросите себя:** как `add_messages` решает, дописать сообщение или заменить, и от чего зависит это решение?
**В следующий раз:** `create_agent` собирает такой граф одним вызовом; затем хуки middleware и проектирование инструментов.